In [2]:
# gj
import json
import sys

# 方法1：直接加载整个文件
with open('/mnt/datasets/liyaqi/MSRVTT/annotation/MSR_VTT.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 获取视频数量
video_count = len(data['images'])
print(f"视频总数: {video_count}")

# 检查是否有caption信息
if 'annotations' in data:
    caption_count = len(data['annotations'])
    print(f"描述总数: {caption_count}")
else:
    print("未找到描述信息")


视频总数: 10000
描述总数: 199994


In [3]:


# 统计 caption 条数
captions = data.get('annotations', [])
caption_count = len(captions)

print(captions[:10])

# 计算所有 caption 的总内存占用（单位：字节）
total_size = sum(sys.getsizeof(c['caption']) for c in captions)

# 计算平均每条 caption 的内存占用（单位：字节）
avg_size = total_size / caption_count if caption_count else 0

print(f"总 caption 条数: {caption_count}")
print(f"总 caption 内存占用: {total_size / 1024:.2f} KB")
print(f"平均每条 caption 占用: {avg_size:.2f} 字节")




[{'caption': 'a cartoon animals runs through an ice cave in a video game', 'id': 0, 'image_id': 'video2960'}, {'caption': 'a cartoon character runs around inside of a video game', 'id': 1, 'image_id': 'video2960'}, {'caption': 'a character is running in the snow', 'id': 2, 'image_id': 'video2960'}, {'caption': 'a person plays a video game centered around ice age the movie', 'id': 3, 'image_id': 'video2960'}, {'caption': 'a person plays online and records themselves', 'id': 4, 'image_id': 'video2960'}, {'caption': 'a scene from the ice age video game is shown', 'id': 5, 'image_id': 'video2960'}, {'caption': 'a video game character is jumping about in a cave', 'id': 6, 'image_id': 'video2960'}, {'caption': 'a video game of a little animal running through an ice tunnel', 'id': 7, 'image_id': 'video2960'}, {'caption': 'a video game of a small animal', 'id': 8, 'image_id': 'video2960'}, {'caption': 'a video shows gameplay from ice age', 'id': 9, 'image_id': 'video2960'}]
总 caption 条数: 19999

In [6]:
print(data.keys())
for e in data.keys():
    print(f"\n=== {e} ===")
    if isinstance(data[e], list):
        print("元素个数:", len(data[e]))
        print("前3个元素:", data[e][:3])
    else:
        print("值类型:", type(data[e]))
        print("内容:", data[e])


dict_keys(['info', 'images', 'licenses', 'type', 'annotations'])

=== info ===
值类型: <class 'dict'>
内容: {'contributor': 'Microsoft MSM group', 'version': '1.0', 'year': '2016', 'data_created': '2016-04-14 14:30:20', 'description': 'This is 1.0 version of the 2016 MSR-VTT dataset.'}

=== images ===
元素个数: 10000
前3个元素: [{'id': 'video0'}, {'id': 'video1'}, {'id': 'video2'}]

=== licenses ===
元素个数: 8
前3个元素: [{'url': 'http://creativecommons.org/licenses/by-nc-sa/2.0/', 'id': 1, 'name': 'Attribution-NonCommercial-ShareAlike License'}, {'url': 'http://creativecommons.org/licenses/by-nc/2.0/', 'id': 2, 'name': 'Attribution-NonCommercial License'}, {'url': 'http://creativecommons.org/licenses/by-nc-nd/2.0/', 'id': 3, 'name': 'Attribution-NonCommercial-NoDerivs License'}]

=== type ===
值类型: <class 'str'>
内容: captions

=== annotations ===
元素个数: 199994
前3个元素: [{'caption': 'a cartoon animals runs through an ice cave in a video game', 'id': 0, 'image_id': 'video2960'}, {'caption': 'a cartoon character

In [7]:
import pandas as pd
import ast
import os
from PIL import Image
from torchvision import transforms

df = pd.DataFrame(data["annotations"])


In [19]:

print(df.head(50))
print(1)
print(df.describe())
print(1)
print(df.columns)
print(1)
captions = df["caption"].tolist()
images = df["image_id"].tolist()
print(type(captions))
print(1)
print(captions[:3])
print(1)
print(images[:3])
print(1)
print(len(captions))

                                              caption  id   image_id
0   a cartoon animals runs through an ice cave in ...   0  video2960
1   a cartoon character runs around inside of a vi...   1  video2960
2                  a character is running in the snow   2  video2960
3   a person plays a video game centered around ic...   3  video2960
4        a person plays online and records themselves   4  video2960
5        a scene from the ice age video game is shown   5  video2960
6   a video game character is jumping about in a cave   6  video2960
7   a video game of a little animal running throug...   7  video2960
8                      a video game of a small animal   8  video2960
9                 a video shows gameplay from ice age   9  video2960
10                a weasel runs around in an icy cave  10  video2960
11  an animal runs through an icy tunnel in a vide...  11  video2960
12           crash and eddie on an ice age video game  12  video2960
13         gameplay footage of som

In [20]:
from collections import Counter

# 假设 images 已经是一个列表
counts = Counter(images)

# 找出出现次数 != 20 的元素
not_20 = {k: v for k, v in counts.items() if v != 20}

print(f"总共有 {len(not_20)} 个元素出现次数不等于 20")
for k, v in list(not_20.items())[:10]:  # 只打印前 10 个
    print(k, v)


总共有 6 个元素出现次数不等于 20
video9268 19
video8605 19
video9896 19
video7360 19
video7851 19
video7512 19


In [23]:

res = {"image_path":[],"image_shape":[],"image_tensor_size":[],"image_file_size":[],"compression_rate":[],"image_caption":[],"image_caption_size":[]}

image_dir = "/mnt/datasets/liyaqi/MSRVTT/videos/all"

for i in range(len(captions)):
    if i % round(len(captions)/4) == 0:
        print(i/len(captions))

    image_path = images[i]+".mp4"
    image_full_path = os.path.join(image_dir, image_path)
    res["image_path"].append(image_path)
    # with Image.open(image_full_path) as img:
    #     image_shape = (len(img.getbands()), img.height, img.width)
    #     tensor = transform(img)
    #     image_tensor_size = tensor.numel() * tensor.element_size()
    # res["image_shape"].append(image_shape)
    res["image_shape"].append(0)
    # res["image_tensor_size"].append(image_tensor_size)
    res["image_tensor_size"].append(0)
    image_file_size = os.path.getsize(image_full_path)
    res["image_file_size"].append(image_file_size)
    # res["compression_rate"].append(f"{image_tensor_size/image_file_size:0.2f}")
    res["compression_rate"].append(0)
    image_caption = captions[i]
    res["image_caption"].append(image_caption)
    image_caption_size = len(image_caption)
    res["image_caption_size"].append(image_caption_size)

out = pd.DataFrame(res)
# out.to_csv("MSRVTT_sample.csv",index=False)

# 提取数字并转换为整数类型
out['numeric_id'] = out['image_path']

# 按数字升序排序
sorted_df = out.sort_values(by='numeric_id', ascending=True)

# 生成新CSV文件（删除辅助列）
sorted_df.drop(columns=['numeric_id']).to_csv('/home/llm/experiment/dataset_Analysis_Modeling/processed_datasets/MSRVTT_sort.csv', index=False)



0.0
0.24999749992499776
0.4999949998499955
0.7499924997749933
0.999989999699991
